In [1]:
from pathlib import Path 
import os 
import sys
import matplotlib.pyplot as plt 
import scanpy as sc
import numpy as np 
import pandas as pd
import scipy as sp 
import seaborn as sns
from tqdm import tqdm 
import copy
import math
import yaml
import fpsample

## Util function 

In [2]:
def manual_filtering(df_dict, 
                     bounds, 
                     columns_to_keep, 
                     celltype_fraction, 
                     margin_frac_bounds, 
                     protocol_cols):
    
    # Filtered data frame 
    df_dict_filtered = {}
    df_dict_annotated = {}
    
    # Iterate over cell types 
    for cell_type in df_dict:
        # Collect cell type dictionary and sort by proportion 
        cell_type_revert_safe_string = cell_type.replace(":", "/")
        df_ct = df_dict[cell_type].copy()
        df_ct = df_ct.sort_values(by=f"{cell_type_revert_safe_string}_prop", ascending=False)
        
        # Subset data frame 
        columns_to_keep_ct = columns_to_keep + [f"{cell_type_revert_safe_string}_prop"]
        df_ct = df_ct.loc[:, columns_to_keep_ct]
        df_ct["filtering_step"] = "pass" 

        # Filter by fraction
        idx_lower = df_ct[f"{cell_type_revert_safe_string}_prop"] < celltype_fraction[cell_type_revert_safe_string] 
        df_ct.loc[idx_lower & (df_ct["filtering_step"] == "pass"), 
                  "filtering_step"] = "proportion_filter"

        # Oxygen and days filtering 
        idx_pass_oxygen_days = np.logical_and(df_ct["o2_[%]"] > 5, 
                                             df_ct["o2_[%]"] < 25)
        idx_pass_oxygen_days = np.logical_and(idx_pass_oxygen_days, 
                                             df_ct["days_of_culture"] > 12)
        idx_pass_oxygen_days = np.logical_and(idx_pass_oxygen_days, 
                                             df_ct["days_of_culture"] < 20)
        df_ct.loc[~idx_pass_oxygen_days & (df_ct["filtering_step"] == "pass"), 
                  "filtering_step"] = "oxygen_days"

        # Bounds 
        for param in protocol_cols:
            idx_bounds = np.logical_and(
                df_ct[param] >= (bounds[param][0] - bounds[param][0] * margin_frac_bounds), 
                df_ct[param] <= (bounds[param][1] + bounds[param][1] * margin_frac_bounds)
            )
            df_ct.loc[~idx_bounds & (df_ct["filtering_step"] == "pass"), 
                      "filtering_step"] = f"bounds_{param}"

        # Filtered dataset
        df_dict_filtered[cell_type] = df_ct[df_ct.filtering_step == "pass"]
        df_dict_annotated[cell_type] = df_ct[df_ct.filtering_step != "proportion_filter"]
        
    return df_dict_filtered, df_dict_annotated

## Read annotation file

In [3]:
ANNOTATION_CFG_FILE = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/annotation/default.yaml"
with open(ANNOTATION_CFG_FILE, "r") as fb:
    annotation_cfg = yaml.safe_load(fb)
protocol_cols = annotation_cfg["protocol_columns"]

In [4]:
ANNOTATION_CFG_FILE_BOUNDS = "/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/collab-goettgens-SFC/inverse/loss_guidance/config/constraints/default_all_cols.yaml"
with open(ANNOTATION_CFG_FILE_BOUNDS, "r") as fb:
    annotation_cfg_bound = yaml.safe_load(fb)
ubound_dict, lbound_dict = annotation_cfg_bound["ubound_dict"],  annotation_cfg_bound["lbound_dict"]

In [5]:
ubound_dict

{'gm-csf_[ng_ml]': 10.0,
 'tpo_[ng_ml]': 2.5,
 'um171_[nm]': 1120.0,
 'um729_[µm]': 1120.0,
 'sr1_[nm]': 750.0,
 'scf_[ng_ml]': 50.0,
 'butyzamide_[nm]': 100.0,
 'retinoic_acid_[µm]': 25.0,
 'ldl_[ng_ml]': 50.0,
 'il3_[ng_ml]': 20.0,
 'o2_[%]': 20.0,
 'days_of_culture': 18.0}

In [6]:
lbound_dict

{'gm-csf_[ng_ml]': 0.0,
 'tpo_[ng_ml]': 0.0,
 'um171_[nm]': 0.0,
 'um729_[µm]': 0.0,
 'sr1_[nm]': 0.0,
 'scf_[ng_ml]': 0.0,
 'butyzamide_[nm]': 0.0,
 'retinoic_acid_[µm]': 0.0,
 'ldl_[ng_ml]': 0.0,
 'il3_[ng_ml]': 0.0,
 'o2_[%]': 10.0,
 'days_of_culture': 12.0}

In [7]:
# Collect per parameter bounds 
bounds = {}
for param in lbound_dict:
    bounds[param] = (lbound_dict[param], ubound_dict[param])

bounds

{'gm-csf_[ng_ml]': (0.0, 10.0),
 'tpo_[ng_ml]': (0.0, 2.5),
 'um171_[nm]': (0.0, 1120.0),
 'um729_[µm]': (0.0, 1120.0),
 'sr1_[nm]': (0.0, 750.0),
 'scf_[ng_ml]': (0.0, 50.0),
 'butyzamide_[nm]': (0.0, 100.0),
 'retinoic_acid_[µm]': (0.0, 25.0),
 'ldl_[ng_ml]': (0.0, 50.0),
 'il3_[ng_ml]': (0.0, 20.0),
 'o2_[%]': (10.0, 20.0),
 'days_of_culture': (12.0, 18.0)}

### Loop over solutions 

In [8]:
path_to_results = Path("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/raw_data")
paths = os.listdir(path_to_results)

In [9]:
cell_types = ["CD14-Mono",
              "CyclingProgenitor*",
              "DCs-CD14",
              "Early_Pro-B", 
              "EosBasoMastPre", 
              "EryPro",
              "GMP-Neutro:CD16-Mono",
              "HSCs",
              "MgkPro",
              "Pro-B"]

In [ ]:
pure_cell_type_solution_dict = {cell_type: [] 
                                for cell_type in cell_types
                               }

for experiment_folder in tqdm(paths):
    # Only consider pure populations 
    if "pure" in experiment_folder:
        cell_type_file_paths = path_to_results / experiment_folder
        # For all cell types 
        for cell_type in os.listdir(cell_type_file_paths):
            config_exp_folders = cell_type_file_paths / cell_type
            # For all the experiments in the folders 
            for config_exp_folder in os.listdir(config_exp_folders):
                # Be sure candidates are there 
                if "candidates.csv" in os.listdir(config_exp_folders / config_exp_folder):
                    uncertainty_path = config_exp_folders / config_exp_folder / "uncertainty_annotation" / "candidates_with_uncertainties.csv"
                    # Be sure uncertainty annotation exists 
                    if uncertainty_path.exists():
                        candidate_with_uncertainty_df = pd.read_csv(uncertainty_path)
                        candidate_with_uncertainty_df["run_name"] = [file.split("/")[-1] for file in candidate_with_uncertainty_df["cfg:paths:dump_dir"]]
                        pure_cell_type_solution_dict[cell_type].append(candidate_with_uncertainty_df)

  0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
# Concatenate data frame 
pure_cell_type_solution_dict_concat = {
    key: pd.concat(val, axis=0) for key, val in pure_cell_type_solution_dict.items()
}

In [ ]:
for col in pure_cell_type_solution_dict_concat["HSCs"].columns:
    print(col)

In [ ]:
# Cell type fractions
celltype_fraction = {
    "MgkPro": 0.15,
    "HSCs": 0.15,
    "EryPro": 0.15,
    "GMP-Neutro/CD16-Mono": 0.15,
    "CyclingProgenitor*": 0.15,
    "EosBasoMastPre": 0.15,
    "Early_Pro-B": 0.15,
    "Pro-B": 0.15,
    "CD14-Mono": 0.15,
    "DCs-CD14": 0.15
}

columns_of_interest = [
    'gm-csf_[ng_ml]',
    'tpo_[ng_ml]',
    'sr1_[nm]',
    'um171_[nm]',
    'um729_[µm]',
    'scf_[ng_ml]',
    'butyzamide_[nm]',
    'retinoic_acid_[µm]',
    'ldl_[ng_ml]',
    'il3_[ng_ml]',
    'o2_[%]',
    'days_of_culture',
    'run_name',
    "cfg:constraints:c_scheduler_kwargs:t_warmup",
    "cfg:constraints:c_scheduler_kwargs:vmin",
    "cfg:constraints:c_scheduler_kwargs:vmax",
    "loss", 
    "target_ct_loss_std",
    "target_ct_loss_mean",
    "ct_prop_total_variance",
    "exploitation_score",
    "exploration_score",
    "weighted_uncertainty_score",
    "metric_response_surface"
]

### Filter values

In [ ]:
# Collect filtered and annotated data frames 
filtered_df, annotated_df = manual_filtering(
    df_dict=pure_cell_type_solution_dict_concat,
    bounds=bounds,
    columns_to_keep=columns_of_interest, 
    celltype_fraction=celltype_fraction,
    margin_frac_bounds=0.5, 
    protocol_cols=protocol_cols
)

In [ ]:
# Filtered and unfiltered solutions 
for ct in filtered_df:
    print(f"Filtered {ct} number of solutions:", filtered_df[ct].shape[0])
    print(f"Unfiltered {ct} number of solutions:", pure_cell_type_solution_dict_concat[ct].shape[0], "\n")

## Downstream

In [ ]:
for cell_type in filtered_df:
    # Cell type safe
    cell_type_safe = cell_type.replace(":", "/")
    
    n_params = len(protocol_cols)
    ncols = 3  # adjust as you like
    nrows = math.ceil(n_params / ncols)

    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(3*ncols, 2*nrows))
    axes = axes.flatten()  # make indexing easy

    for i, param in enumerate(protocol_cols):
        sns.scatterplot(
            data=filtered_df[cell_type],
            x=param,
            y=f"{cell_type_safe}_prop",
            ax=axes[i], 
            s=10
        )
        axes[i].set_title(f"{cell_type} vs {param}")

    # Remove unused axes if any
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

## Check the calibration (loss vs loss_std)

In [ ]:
sns.set(style="whitegrid", context="paper")

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13,7))
fig.subplots_adjust(
    hspace=0.6,   # vertical space between rows
    wspace=0.4    # horizontal space between columns
)
axes = axes.flatten()  # make indexing easy

for i, cell_type in enumerate(pure_cell_type_solution_dict_concat):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = pure_cell_type_solution_dict_concat[cell_type].reset_index(drop=True)

    ax = axes[i]

    sns.scatterplot(
        data=df_plot,
        x=f"{cell_type_safe}_prop",
        # x="target_ct_loss_mean",
        y="target_ct_loss_std",
        ax=ax
    )

    # ax.set_yscale("log")   
    ax.set_title(cell_type)
    ax.set_xlabel("Cell type proportion")
    ax.set_ylabel("Target CT loss (std)")
    sns.despine(ax=ax)

In [ ]:
sns.set(style="whitegrid", context="paper")

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13,7))
fig.subplots_adjust(
    hspace=0.6,   # vertical space between rows
    wspace=0.4    # horizontal space between columns
)
axes = axes.flatten()  # make indexing easy

for i, cell_type in enumerate(pure_cell_type_solution_dict_concat):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = pure_cell_type_solution_dict_concat[cell_type].reset_index(drop=True)

    ax = axes[i]

    sns.scatterplot(
        data=df_plot,
        x="target_ct_loss_mean",
        y="target_ct_loss_std",
        ax=ax
    )

    # ax.set_yscale("log")   
    ax.set_title(cell_type)
    ax.set_xlabel("Cell type proportion")
    ax.set_ylabel("Target CT loss (std)")
    sns.despine(ax=ax)

In [ ]:
sns.set(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.3)  # slightly more compact spacing
axes = axes.flatten()

# Define a color palette for 'filtering_step'
palette = sns.color_palette("Set2")  # soft but distinguishable colors

for i, cell_type in enumerate(annotated_df):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = annotated_df[cell_type].reset_index(drop=True)

    ax = axes[i]

    # scatter with hue, small and semi-transparent points
    sns.scatterplot(
        data=df_plot,
        x=f"{cell_type_safe}_prop",
        y="target_ct_loss_std",
        hue="filtering_step",
        palette=palette,
        s=35,
        alpha=0.7,
        ax=ax,
        edgecolor=None
    )

    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("Cell type proportion", fontsize=9)
    ax.set_ylabel("Target CT loss (std)", fontsize=9)
    ax.legend_.remove()  # remove individual legends to declutter
    sns.despine(ax=ax)

# Add a single legend for the figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False, fontsize=10)

plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.3)
axes = axes.flatten()

for i, cell_type in enumerate(annotated_df):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = annotated_df[cell_type].reset_index(drop=True)
    df_plot["pass_filter"] = (df_plot["filtering_step"] == "pass")

    ax = axes[i]

    # Map True/False to colors
    color_map = {True: "firebrick", False: "lightgray"}
    alpha_map = {True: 1, False: 0.4}  # more transparent for False points

    for val in [False, True]:
        subset = df_plot[df_plot["pass_filter"] == val]
        ax.scatter(
            subset[f"{cell_type_safe}_prop"],
            subset["target_ct_loss_std"],
            color=color_map[val],
            alpha=alpha_map[val],
            s=35,
            edgecolor=None,
            label=str(val)
        )

    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("Cell type proportion", fontsize=9)
    ax.set_ylabel("Target CT loss (std)", fontsize=9)
    sns.despine(ax=ax)

# Add a single legend for the figure
fig.legend(
    ["fail", "pass"],  # nicer labels
    loc="upper center",
    ncol=2,
    frameon=False,
    fontsize=10
)

plt.show()

In [ ]:
sns.set(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 8))
fig.subplots_adjust(hspace=0.8, wspace=0.5)
axes = axes.flatten()

for i, cell_type in enumerate(annotated_df):
    df_plot = annotated_df[cell_type].reset_index(drop=True)
    ax = axes[i]

    # Compute counts for sorting
    counts = df_plot["filtering_step"].value_counts()
    sorted_categories = counts.index.tolist()  # ordered by count descending

    # Count plot ordered by height
    sns.countplot(
        data=df_plot,
        x="filtering_step",
        order=sorted_categories,
        ax=ax
    )

    # Rotate x-axis labels
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yscale("log")  # log scale
    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("")  # optional: remove repeated x-labels
    ax.set_ylabel("Count", fontsize=9)
    sns.despine(ax=ax)

# Remove empty subplots if fewer than 10
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.show()

## Rearrange solutions' columns 

In [ ]:
for cell_type in filtered_df:
    # Clean filtered data frames 
    filtered_df[cell_type]= filtered_df[cell_type].rename({"target_ct_loss_std": "uncertainty_loss", 
                                                           "ct_prop_total_variance": "uncertainty_total_variance_ct"}, 
                                                          axis=1)
    
    filtered_df[cell_type] = filtered_df[cell_type].drop(["cfg:constraints:c_scheduler_kwargs:t_warmup",
                                                          "cfg:constraints:c_scheduler_kwargs:vmin",
                                                          "cfg:constraints:c_scheduler_kwargs:vmax"], axis=1)

    # Clean unfiltered dataframes 
    annotated_df[cell_type]= annotated_df[cell_type].rename({"target_ct_loss_std": "uncertainty_loss", 
                                                           "ct_prop_total_variance": "uncertainty_total_variance_ct"}, 
                                                          axis=1)
    
    annotated_df[cell_type] = annotated_df[cell_type].drop(["cfg:constraints:c_scheduler_kwargs:t_warmup",
                                                          "cfg:constraints:c_scheduler_kwargs:vmin",
                                                          "cfg:constraints:c_scheduler_kwargs:vmax"], axis=1)

## Subsample by uncertainty 

In [ ]:
# filtered_df_subsample = {}

# for cell_type in filtered_df:
#     if filtered_df[cell_type].shape[0] > 200:
#         filtered_df_subsample_ct = []
#         sorted_df_uncertainty = filtered_df[cell_type].sort_values(by="uncertainty_loss", ascending=True)
#         filtered_df_subsample_ct.append(sorted_df_uncertainty[:100])
#         filtered_df_subsample_ct.append(sorted_df_uncertainty[-100:])
#         filtered_df_subsample_ct = pd.concat(filtered_df_subsample_ct, axis=0)
#         filtered_df_subsample[cell_type] = filtered_df_subsample_ct
#     else:
#         filtered_df_subsample[cell_type] = filtered_df[cell_type]

In [ ]:
# for cell_type in filtered_df_subsample:
#     print(cell_type)
#     print(filtered_df_subsample[cell_type].shape)

In [ ]:
# sns.set(style="whitegrid", context="paper")

# fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13,7))
# fig.subplots_adjust(
#     hspace=0.6,   # vertical space between rows
#     wspace=0.4    # horizontal space between columns
# )
# axes = axes.flatten()  # make indexing easy

# for i, cell_type in enumerate(filtered_df_subsample):
#     cell_type_safe = cell_type.replace(":", "/")
#     df_plot = filtered_df_subsample[cell_type].reset_index(drop=True)

#     ax = axes[i]

#     sns.scatterplot(
#         data=df_plot,
#         x=f"{cell_type_safe}_prop",
#         y="uncertainty_loss",
#         ax=ax   
#     )

#     # ax.set_yscale("log")   
#     ax.set_title(cell_type)
#     ax.set_xlabel("Cell type proportion")
#     ax.set_ylabel("Target CT loss (std)")
#     sns.despine(ax=ax)

## Cluster and reduce number of solutions 

In [ ]:
# np.random.seed(42)

# filtered_df_subsampled = {}
# annotated_df_subsampled = {}

# n_samples_filtered = 100
# n_samples_unfiltered = 100

# def fps_subsample(df, cols, n):
#     if df.shape[0] <= n:
#         return df

#     X = df[cols].values
#     sampled_ids = fpsample.fps_sampling(X, n)
#     return df.iloc[sampled_ids]

# for cell_type in tqdm(filtered_df):

#     filtered_df_subsampled[cell_type] = fps_subsample(
#         filtered_df[cell_type],
#         protocol_cols,
#         n_samples_filtered
#     )

#     annotated_df_subsampled[cell_type] = fps_subsample(
#         annotated_df[cell_type],
#         protocol_cols,
#         n_samples_unfiltered
#     )

## Save results

In [ ]:
# RESULT_PATH = Path("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/inverse/loss_guidance/manually_filtered")
# RESULT_PATH_FILTERED = RESULT_PATH / "filtered"
# RESULT_PATH_FILTERED.mkdir(exist_ok=True, parents=True)
# # RESULT_PATH_ANNOTATED = RESULT_PATH / "annotated"
# # RESULT_PATH_ANNOTATED.mkdir(exist_ok=True, parents=True)

# # Save results 
# for cell_type in filtered_df_subsample:
#     filtered_df_subsample[cell_type].to_csv(RESULT_PATH / "filtered" / f"filtered_candidates_{cell_type}.csv")
#     # annotated_df[cell_type].to_csv(RESULT_PATH / "annotated" / f"annotated_candidates_{cell_type}.csv")

## Check if uncertainty is connected to rarity

In [ ]:
# Read the annotated dataset
adata = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/data/gdrive/ds_HID01_logicle_100k_UMAP_ann.h5ad")

In [ ]:
ct_proportions = adata.obs.cell_type.value_counts() /  adata.shape[0]
annotated_df_concat = annotated_df.copy()

In [ ]:
for cell_type in annotated_df_concat:
    cell_type_safe = cell_type.replace(":", "/")
    annotated_df_concat[cell_type]["real_ct_prop"] = ct_proportions[cell_type_safe]
    annotated_df_concat[cell_type]["ct"] = cell_type

In [ ]:
annotated_df_concat = pd.concat(list(annotated_df_concat.values()), axis=0)
annotated_df_concat = annotated_df_concat.reset_index()

In [ ]:
# Style
sns.set(style="whitegrid")

plt.figure(figsize=(4,4))

ax = sns.boxplot(
    data=annotated_df_concat,
    x="ct",
    y="uncertainty_loss",
    palette="viridis",
    showfliers=False   # optional: hide extreme outliers
)


ax.set_title("Uncertainty Loss by CT Category")
ax.set_xlabel("CT")
ax.set_ylabel("Uncertainty Loss")

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
sns.set(style="whitegrid", context="talk")

# Get unique ct_prop per ct
ct_prop_df = (
    annotated_df_concat[["ct", "real_ct_prop"]]
    .drop_duplicates()
)

# Sort cell types by proportion
ct_order = (
    ct_prop_df
    .sort_values("real_ct_prop", ascending=False)["ct"]
    .tolist()
)

fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(9,9),
    sharex=True,
)

# Boxplot (top)
sns.boxplot(
    data=annotated_df_concat,
    x="ct",
    y="uncertainty_loss",
    palette="viridis",
    showfliers=False,
    order=ct_order,
    ax=ax1
)

ax1.set_title("Uncertainty Loss by CT Category")
ax1.set_ylabel("Uncertainty Loss")
ax1.set_xlabel("")

# Barplot (bottom)
sns.barplot(
    data=ct_prop_df,
    x="ct",
    y="real_ct_prop",
    color="gray",
    order=ct_order,
    ax=ax2
)

ax2.set_ylabel("CT Prop")
ax2.set_xlabel("CT")

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## Are the uncertain protocols different from the others 

In [ ]:
unique_protocols = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/miscellaneous/unique_concentrations_adata.h5ad")

In [ ]:
for cell_type, df in pure_cell_type_solution_dict_concat.items():

    X_ct_generated = df.loc[:, protocol_cols].values

    C = np.mean(
        (np.log1p(X_ct_generated[:, None, :]) - unique_protocols.X[None, :, :])**2,
        axis=-1
    )

    min_dist = C.min(axis=1)
    mean_dist = C.mean(axis=1)
    max_dist = C.max(axis=1)

    # Save distances
    df.loc[:, "min_dist_from_real"] = min_dist
    df.loc[:, "mean_dist_from_real"] = mean_dist
    df.loc[:, "max_dist_from_real"] = max_dist

In [ ]:
sns.set(style="whitegrid", context="paper")

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13,7))
fig.subplots_adjust(
    hspace=0.6,   # vertical space between rows
    wspace=0.4    # horizontal space between columns
)
axes = axes.flatten()  # make indexing easy

for i, cell_type in enumerate(pure_cell_type_solution_dict_concat):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = pure_cell_type_solution_dict_concat[cell_type].reset_index(drop=True)

    ax = axes[i]

    sns.scatterplot(
        data=df_plot,
        x="mean_dist_from_real",
        y="target_ct_loss_std",
        ax=ax   
    )

    # ax.set_yscale("log")   
    ax.set_title(cell_type)
    ax.set_xlabel("Min distance from real")
    ax.set_ylabel("Target CT loss (std)")
    sns.despine(ax=ax)

# Check the uncertainty measures by Lorenzo 

In [ ]:
sns.set(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.3)  # slightly more compact spacing
axes = axes.flatten()

for i, cell_type in enumerate(annotated_df):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = annotated_df[cell_type].reset_index(drop=True)

    ax = axes[i]

    # scatter with hue, small and semi-transparent points
    sns.scatterplot(
        data=df_plot,
        x="exploration_score",
        y="uncertainty_loss",
        hue="target_ct_loss_mean",
        palette="viridis",
        s=35,
        alpha=0.7,
        ax=ax,
        edgecolor=None
    )

    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("Exploration score", fontsize=9)
    ax.set_ylabel("Target CT loss (std)", fontsize=9)
    ax.legend_.remove()  # remove individual legends to declutter
    sns.despine(ax=ax)

# Add a single legend for the figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False, fontsize=10)

plt.show()

In [ ]:
sns.set(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.3)  # slightly more compact spacing
axes = axes.flatten()

for i, cell_type in enumerate(annotated_df):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = annotated_df[cell_type].reset_index(drop=True)

    ax = axes[i]

    # scatter with hue, small and semi-transparent points
    sns.scatterplot(
        data=df_plot,
        x="exploitation_score",
        y="uncertainty_loss",
        hue="target_ct_loss_mean",
        palette="viridis",
        s=35,
        alpha=0.7,
        ax=ax,
        edgecolor=None
    )

    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("Exploitation score", fontsize=9)
    ax.set_ylabel("Target CT loss (std)", fontsize=9)
    ax.legend_.remove()  # remove individual legends to declutter
    sns.despine(ax=ax)

# Add a single legend for the figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False, fontsize=10)

plt.show()

In [ ]:
sns.set(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.3)  # slightly more compact spacing
axes = axes.flatten()

for i, cell_type in enumerate(annotated_df):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = annotated_df[cell_type].reset_index(drop=True)

    ax = axes[i]

    # scatter with hue, small and semi-transparent points
    sns.scatterplot(
        data=df_plot,
        x="weighted_uncertainty_score",
        y="uncertainty_loss",
        hue="target_ct_loss_mean",
        palette="viridis",
        s=35,
        alpha=0.7,
        ax=ax,
        edgecolor=None
    )

    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("Exploitation score", fontsize=9)
    ax.set_ylabel("Target CT loss (std)", fontsize=9)
    ax.legend_.remove()  # remove individual legends to declutter
    sns.despine(ax=ax)

# Add a single legend for the figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False, fontsize=10)

plt.show()

In [ ]:
sns.set(style="whitegrid", context="paper", font_scale=1.1)

fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(13, 7))
fig.subplots_adjust(hspace=0.5, wspace=0.3)  # slightly more compact spacing
axes = axes.flatten()

for i, cell_type in enumerate(annotated_df):
    cell_type_safe = cell_type.replace(":", "/")
    df_plot = annotated_df[cell_type].reset_index(drop=True)

    ax = axes[i]

    # scatter with hue, small and semi-transparent points
    sns.scatterplot(
        data=df_plot,
        x="metric_response_surface",
        y="uncertainty_loss",
        hue="target_ct_loss_mean",
        palette="viridis",
        s=35,
        alpha=0.7,
        ax=ax,
        edgecolor=None
    )

    ax.set_title(cell_type, fontsize=10)
    ax.set_xlabel("Metric response curve", fontsize=9)
    ax.set_ylabel("Target CT loss (std)", fontsize=9)
    ax.legend_.remove()  
    sns.despine(ax=ax)

# Add a single legend for the figure
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=len(labels), frameon=False, fontsize=10)

plt.show()

## Check if the closest protocol to Sakurai is in the uncertainty pool 

In [ ]:
sakurai_medium = np.array([0, 0, 0, 70, 0, 0, 100, 0, 0, 0, 20, 14])

In [ ]:
MgkPro_df = filtered_df["MgkPro"]
MgkPro_df = MgkPro_df.reset_index()

In [ ]:
dist_mgkpro_sakurai = np.abs(MgkPro_df.loc[:, protocol_cols].values - sakurai_medium[None, :]).sum(1)

In [ ]:
min_dist_sakurai = np.argmin(dist_mgkpro_sakurai)

In [ ]:
MgkPro_df.iloc[min_dist_sakurai]["uncertainty_loss"]

In [ ]:
sns.kdeplot(MgkPro_df, x="uncertainty_loss")
plt.axvline(MgkPro_df.iloc[min_dist_sakurai]["uncertainty_loss"], linestyle="--", linewidth=2)

In [ ]:
min_dist_sakurai in MgkPro_df.sort_values("uncertainty_loss", ascending=False)[:100].index

In [ ]:
MgkPro_df["MgkPro_prop"].max()

## Check uncertainties at every filtering step 

In [80]:
# def annotate_filtering_steps(): 

In [81]:
# for cell_type in pure_cell_type_solution_dict_concat:
#     cell_type_safe = cell_type.replace(":", "/")

#     df_plot = pure_cell_type_solution_dict_concat[cell_type].reset_index(drop=True)

#     plt.figure(figsize=(2,2))
#     sns.scatterplot(
#         data=df_plot,
#         x=f"{cell_type_safe}_prop",
#         y="target_ct_loss_std",
#         edgecolor="black"
#     )
#     plt.show()

## Compare uncertainty of solutions that make the cut 

In [104]:
# for cell_type in annotated_df:

#     df_plot = annotated_df[cell_type].reset_index(drop=True)

#     plt.figure(figsize=(2.8, 3.2))

#     ax = sns.boxplot(
#         data=df_plot,
#         x="filtering_step",
#         y="target_ct_loss_std",
#         hue="filtering_step", 
#         order=["proportion_filter", "oxygen_days", "bounds", "pass"],
#         width=0.6,
#         fliersize=1,
#         palette="colorblind"
#     )

#     # Clean aesthetics
#     sns.despine()
#     ax.set_xlabel("")
#     ax.set_ylabel("Target CT loss (std)")
#     ax.set_title(cell_type, fontsize=10)

#     plt.xticks(rotation=90, ha="right")
#     plt.tight_layout()
#     plt.show()

TODO:
* Compare with Sakurai
* Same analysis with custom (HSCs, MgkPro, EryPro)
* Uncertainty calibration 

In [72]:
# annotated_df["Early_Pro-B"]["filtering_step"].value_counts()

In [73]:
# annotated_df["HSCs"]["filtering_step"].value_counts()

In [74]:
# annotated_df["Early_Pro-B"].max(0)

In [75]:
# bounds

In [76]:
# for t_warmup, vmin, vmax in zip(filtered_df["Early_Pro-B"]["cfg:constraints:c_scheduler_kwargs:t_warmup"], filtered_df["Early_Pro-B"]["cfg:constraints:c_scheduler_kwargs:vmax"], filtered_df["Early_Pro-B"]["cfg:constraints:c_scheduler_kwargs:vmin"]):
#     print("t_warmup:", t_warmup, "vmin:", vmin, "vmax:", vmax)

In [77]:
# for t_warmup, vmin, vmax in zip(filtered_df["HSCs"]["cfg:constraints:c_scheduler_kwargs:t_warmup"], filtered_df["HSCs"]["cfg:constraints:c_scheduler_kwargs:vmax"], filtered_df["Early_Pro-B"]["cfg:constraints:c_scheduler_kwargs:vmin"]):
#     print("t_warmup:", t_warmup, "vmin:", vmin, "vmax:", vmax)

In [78]:
# for t_warmup, vmin, vmax in zip(filtered_df["Pro-B"]["cfg:constraints:c_scheduler_kwargs:t_warmup"], filtered_df["Pro-B"]["cfg:constraints:c_scheduler_kwargs:vmax"], filtered_df["Early_Pro-B"]["cfg:constraints:c_scheduler_kwargs:vmin"]):
#     print("t_warmup:", t_warmup, "vmin:", vmin, "vmax:", vmax)

In [79]:
# for t_warmup, vmin, vmax in zip(filtered_df["EryPro"]["cfg:constraints:c_scheduler_kwargs:t_warmup"], filtered_df["EryPro"]["cfg:constraints:c_scheduler_kwargs:vmax"], filtered_df["EryPro"]["cfg:constraints:c_scheduler_kwargs:vmin"]):
#     print("t_warmup:", t_warmup, "vmin:", vmin, "vmax:", vmax)

In [80]:
# filtered_df["EryPro"]["run_name"].value_counts()

In [81]:
# filtered_df["EryPro"][filtered_df["EryPro"]["EryPro_prop"]>0.40].loc[:, protocol_cols]

## Check values of penalization parameters

In [82]:
# sns.histplot(data=filtered_df["Early_Pro-B"],
#              x="cfg:constraints:c_scheduler_kwargs:t_warmup")
# plt.show()

# sns.histplot(data=filtered_df["Early_Pro-B"],
#              x="cfg:constraints:c_scheduler_kwargs:vmin")
# plt.show()

# sns.histplot(data=filtered_df["Early_Pro-B"],
#              x="cfg:constraints:c_scheduler_kwargs:vmax")
# plt.show()

In [83]:
# sns.histplot(data=filtered_df["HSCs"],
#              x="cfg:constraints:c_scheduler_kwargs:t_warmup")
# plt.show()

# sns.histplot(data=filtered_df["HSCs"],
#              x="cfg:constraints:c_scheduler_kwargs:vmin")
# plt.show()

# sns.histplot(data=filtered_df["HSCs"],
#              x="cfg:constraints:c_scheduler_kwargs:vmax")
# plt.show()

In [84]:
# sns.histplot(data=filtered_df["Pro-B"],
#              x="cfg:constraints:c_scheduler_kwargs:t_warmup")
# plt.show()

# sns.histplot(data=filtered_df["Pro-B"],
#              x="cfg:constraints:c_scheduler_kwargs:vmin")
# plt.show()

# sns.histplot(data=filtered_df["Pro-B"],
#              x="cfg:constraints:c_scheduler_kwargs:vmax")
# plt.show()

## Check HSCs

In [85]:
# hsc = annotated_df["HSCs"]

In [86]:
# hsc = hsc[hsc["HSCs_prop"]>0.20]

In [87]:
# hsc["HSCs_prop"]

In [88]:
# annotated_df["HSCs"]["filtering_step"].value_counts()

In [89]:
# plt.hist(annotated_df["HSCs"]["o2_[%]"], 50)

## Read new dataset

In [61]:
# adata_new = sc.read_h5ad("/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/output/miscellaneous/ds_HID01_logicle_100k_UMAP_ann_with_LPHO12.h5ad")

In [62]:
# for experiment in ["205", "206", "207", "208", "209", "210"]:
#     adata_new_copy = adata_new.copy()
#     sc.pl.umap(adata_new_copy, color="experiment_number", groups=[experiment], palette="r")

In [63]:
# sc.tl.pca(adata_new)
# sc.pp.neighbors(adata_new)
# sc.tl.umap(adata_new)

In [64]:
# for experiment in ["205", "206", "207", "208", "209", "210"]:
#     adata_new_copy = adata_new.copy()
#     sc.pl.umap(adata_new_copy, color="experiment_number", groups=[experiment], palette="r", s=10)

In [65]:
# sc.pl.umap(adata_new_copy, color="cell_type", s=12)